In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install transformers datasets accelerate -q

In [2]:
from datasets import load_dataset
dataset = load_dataset("roneneldan/TinyStories")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [3]:
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

small_train = dataset["train"].select(range(20000))
small_val = dataset["validation"].select(range(2000))
tokenized_train = small_train.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_val = small_val.map(tokenize_fn, batched=True, remove_columns=["text"])

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [4]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [5]:
from transformers import GPT2Config, GPT2LMHeadModel

tiny_config = GPT2Config(vocab_size=tokenizer.vocab_size, n_positions=128, n_embd=128, n_layer=4, n_head=4)
zoom_config = GPT2Config(vocab_size=tokenizer.vocab_size, n_positions=128, n_embd=144, n_layer=4, n_head=4)
xsmall_config = GPT2Config(vocab_size=tokenizer.vocab_size, n_positions=128, n_embd=160, n_layer=4, n_head=4)
small_config = GPT2Config(vocab_size=tokenizer.vocab_size, n_positions=128, n_embd=192, n_layer=6, n_head=6)
medium_config = GPT2Config(vocab_size=tokenizer.vocab_size, n_positions=128, n_embd=256, n_layer=8, n_head=8)

In [6]:
from transformers import AutoModelForCausalLM

teacher_neo = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-125m")
teacher_neo.eval()
teacher_neo.to("cuda")
print(f"GPT-Neo teacher params: {sum(p.numel() for p in teacher_neo.parameters())/1e6:.1f}M")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/526M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125m
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

GPT-Neo teacher params: 125.2M


In [7]:
import torch
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments

class DistillationTrainer(Trainer):
    def __init__(self, teacher_model=None, temperature=2.0, alpha=0.5, **kwargs):
        super().__init__(**kwargs)
        self.teacher = teacher_model
        self.temperature = temperature
        self.alpha = alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        student_loss = outputs.loss
        student_logits = outputs.logits

        with torch.no_grad():
            teacher_outputs = self.teacher(**inputs)
            teacher_logits = teacher_outputs.logits

        T = self.temperature
        student_logits_flat = student_logits.view(-1, student_logits.size(-1))
        teacher_logits_flat = teacher_logits.view(-1, teacher_logits.size(-1))

        distill_loss = F.kl_div(
            F.log_softmax(student_logits_flat / T, dim=-1),
            F.softmax(teacher_logits_flat / T, dim=-1),
            reduction="batchmean"
        ) * (T ** 2)

        loss = self.alpha * student_loss + (1 - self.alpha) * distill_loss
        return (loss, outputs) if return_outputs else loss

def run_distilled_condition(config, teacher_model, seed, args_template, alpha=0.5, temperature=2.0):
    torch.manual_seed(seed)
    model = GPT2LMHeadModel(config)
    model.to("cuda")

    trainer = DistillationTrainer(
        model=model, teacher_model=teacher_model, temperature=temperature, alpha=alpha,
        args=args_template,
        train_dataset=tokenized_train, eval_dataset=tokenized_val, data_collator=data_collator,
    )
    trainer.train()

    plain_eval_trainer = Trainer(
        model=model, args=args_template,
        eval_dataset=tokenized_val, data_collator=data_collator,
    )
    result = plain_eval_trainer.evaluate()
    return result["eval_loss"]

In [8]:
neo_results = {}
configs = {
    "Tiny": tiny_config, "Zoom": zoom_config, "XSmall": xsmall_config,
    "Small": small_config, "Medium": medium_config,
}

for name, cfg in configs.items():
    args = TrainingArguments(
        output_dir=f"./neo_distill_{name}",
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        num_train_epochs=1, logging_steps=200, eval_strategy="steps",
        eval_steps=200, save_strategy="no", report_to="none",
    )
    loss = run_distilled_condition(cfg, teacher_neo, seed=42, args_template=args, alpha=0.5)
    neo_results[name] = loss
    print(f"{name}, GPT-Neo teacher, alpha=0.5: distilled loss = {loss:.3f}")

print(neo_results)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
200,8.627305,7.778415
400,7.229022,6.728021
600,6.430607,6.125655
800,5.972477,5.783665
1000,5.711013,5.598753
1200,5.589320,5.525201


Tiny, GPT-Neo teacher, alpha=0.5: distilled loss = 6.226


Step,Training Loss,Validation Loss
200,8.478873,7.544115
400,6.966880,6.446292
600,6.152350,5.847392
800,5.707113,5.520890
1000,5.460524,5.347666
1200,5.346985,5.279395


Zoom, GPT-Neo teacher, alpha=0.5: distilled loss = 5.939


Step,Training Loss,Validation Loss
200,8.408391,7.397301
400,6.798223,6.262773
600,5.971829,5.666308
800,5.533961,5.347629
1000,5.294868,5.181172
1200,5.185065,5.115422


XSmall, GPT-Neo teacher, alpha=0.5: distilled loss = 5.763


Step,Training Loss,Validation Loss
200,8.154608,7.033647
400,6.398342,5.843002
600,5.548692,5.252125
800,5.131263,4.956335
1000,4.911906,4.803625
1200,4.808626,4.741212


Small, GPT-Neo teacher, alpha=0.5: distilled loss = 5.338


Step,Training Loss,Validation Loss
200,7.718105,6.362055
400,5.690870,5.120898
600,4.876812,4.609527
800,4.517698,4.347322
1000,4.315172,4.203447
1200,4.216067,4.145159


Medium, GPT-Neo teacher, alpha=0.5: distilled loss = 4.700
{'Tiny': 6.226149559020996, 'Zoom': 5.939071178436279, 'XSmall': 5.763278961181641, 'Small': 5.337698936462402, 'Medium': 4.700262546539307}


In [9]:
gpt2_baseline = {"Tiny": 5.850, "Zoom": 5.613, "XSmall": 5.402, "Small": 4.992, "Medium": 4.469}
gpt2_distilled = {"Tiny": 6.258, "Zoom": 5.968, "XSmall": 5.785, "Small": 5.356, "Medium": 4.740}

print(f"{'Size':<10}{'Baseline':<12}{'GPT-2 dist.':<14}{'GPT-2 gap':<12}{'Neo dist.':<12}{'Neo gap':<10}")
for name in configs:
    base = gpt2_baseline[name]
    gpt2_gap = base - gpt2_distilled[name]
    neo_gap = base - neo_results[name]
    print(f"{name:<10}{base:<12.3f}{gpt2_distilled[name]:<14.3f}{gpt2_gap:<12.3f}{neo_results[name]:<12.3f}{neo_gap:<10.3f}")

Size      Baseline    GPT-2 dist.   GPT-2 gap   Neo dist.   Neo gap   
Tiny      5.850       6.258         -0.408      6.226       -0.376    
Zoom      5.613       5.968         -0.355      5.939       -0.326    
XSmall    5.402       5.785         -0.383      5.763       -0.361    
Small     4.992       5.356         -0.364      5.338       -0.346    
Medium    4.469       4.740         -0.271      4.700       -0.231    
